# NumCompute

In [21]:
from pathlib import Path
import sys
import numpy as np

# Supports launching Jupyter from the project root or the demo folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "numcompute" / "__init__.py").is_file()
             and (p / "demo" / "data" / "students.csv").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the NumCompute project root or demo folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numcompute as nc  # type: ignore
from benchmark.bench_integrated import format_table, run_benchmarks  # type: ignore
from IPython.display import Markdown, display

np.set_printoptions(precision=3, suppress=True)
print("Python:", sys.version.split()[0], "| NumPy:", np.__version__)
print("NumCompute:", nc.__version__)


Python: 3.12.13 | NumPy: 2.5.2
NumCompute: 0.1.0


## 1. CSV I/O

A six-student synthetic CSV contains a blank attendance field, tied scores, numeric columns and a categorical group. Explicit dtypes preserve each column. Only hours and attendance enter X; score and passed are kept separate.


In [11]:
print("\n1. CSV I/O - six students, one missing attendance")
csv_path = ROOT / "demo" / "data" / "students.csv"
schema = [("hours", "f8"), ("attendance", "f8"), ("score", "f8"),
          ("group", "U1"), ("passed", "i4")]
data = nc.load_csv(csv_path, dtype=schema, skip_header=1)
X = np.column_stack([data["hours"], data["attendance"]])
scores = data["score"]
y_true = data["passed"]
categories = data["group"].reshape(-1, 1)
print("Features [hours, attendance]:\n", X)
print("Scores:", scores)
print("Missing values:", int(np.isnan(X).sum()))
assert data.shape == (6,) and np.isnan(X).sum() == 1



1. CSV I/O - six students, one missing attendance
Features [hours, attendance]:
 [[ 1. 60.]
 [ 2. 65.]
 [ 3. nan]
 [ 4. 80.]
 [ 5. 90.]
 [ 6. 95.]]
Scores: [60. 78. 85. 92. 92. 95.]
Missing values: 1


## 2. Preprocessing and Pipeline

Impute the missing attendance with the demonstration constant 75, then standardise each feature. Each column has mean approximately 0 and population standard deviation 1. Fit once on the example data; transform new rows using those same fitted statistics.


In [12]:
print("\n2. Preprocessing and Pipeline")
# 75 is a deliberate constant for this synthetic attendance example.
# Targets/scores stay out of X. Fit once, reuse transform on new data.
preprocess = nc.Pipeline([
    ("impute", nc.Imputer(fill_value=75.0)),
    ("scale", nc.StandardScaler()),
])
prepared = preprocess.fit_transform(X)
print("Standardised features:\n", prepared)
print("Column means:", nc.mean(prepared, axis=0))
print("Column standard deviations:", nc.std(prepared, axis=0))
np.testing.assert_allclose(nc.mean(prepared, axis=0), [0, 0], atol=1e-12)
np.testing.assert_allclose(nc.std(prepared, axis=0), [1, 1], atol=1e-12)
learned_mean = preprocess["scale"].mean_.copy()
new_data = np.array([[2.5, 70.0], [6.5, np.nan]])
print("New rows, using the fitted pipeline:\n", preprocess.transform(new_data))
np.testing.assert_array_equal(preprocess["scale"].mean_, learned_mean)



2. Preprocessing and Pipeline
Standardised features:
 [[-1.464 -1.4  ]
 [-0.878 -1.   ]
 [-0.293 -0.2  ]
 [ 0.293  0.2  ]
 [ 0.878  1.   ]
 [ 1.464  1.4  ]]
Column means: [-0.  0.]
Column standard deviations: [1. 1.]
New rows, using the fitted pipeline:
 [[-0.586 -0.6  ]
 [ 1.757 -0.2  ]]


### Additional preprocessing examples

OneHotEncoder maps A/B to two indicator columns. MinMaxScaler maps this example to [0,1]. FeatureUnion joins two transformed branches; appending one-hot columns gives shape (6,6).


In [13]:
clean = preprocess["impute"].transform(X)
encoded = nc.OneHotEncoder().fit_transform(categories)
minmax = nc.MinMaxScaler().fit_transform(clean)
print("Groups A/B, one-hot:\n", encoded)
print("Min-max feature range:", minmax.min(axis=0), minmax.max(axis=0))
union = nc.make_union(nc.StandardScaler(), nc.MinMaxScaler())
combined = np.column_stack([union.fit_transform(clean), encoded])
print("FeatureUnion (two scalings) + one-hot shape:", combined.shape)
assert combined.shape == (6, 6) and np.isfinite(combined).all()


Groups A/B, one-hot:
 [[1 0]
 [0 1]
 [1 0]
 [0 1]
 [1 0]
 [0 1]]
Min-max feature range: [0. 0.] [1. 1.]
FeatureUnion (two scalings) + one-hot shape: (6, 6)


## 3. Sorting, searching and ranking

Top-3 scores are [95, 92, 92]. Stable sorting preserves the order of the two 92s. Ranking is ascending: average ranks share positions 4 and 5 as 4.5; dense ranks assign both scores rank 4. Binary search for 80 returns insertion index 2 and False. Quickselect uses zero-based k, so k=2 returns 85.


In [14]:
print("\n3. Sorting, searching and ranking")
sorted_scores, order = nc.stable_sort(scores, descending=True, return_indices=True)
print("Stable descending scores / indices:", sorted_scores, order)
print("Top-3 scores:", nc.topk(scores, 3, return_indices=False))
print("Third smallest (zero-based k=2):", nc.quickselect(scores, 2, seed=42))
print("Search 80 (insertion index, exists):", nc.binary_search(np.sort(scores), 80))
average_ranks = nc.rank_values(scores, method="average")
dense_ranks = nc.rank_values(scores, method="dense")
print("Average ranks (smallest is 1):", average_ranks)
print("Dense ranks:", dense_ranks)
print("Score quartiles:", nc.percentile(scores, [25, 50, 75]))
np.testing.assert_array_equal(average_ranks, [1, 2, 3, 4.5, 4.5, 6])
np.testing.assert_array_equal(dense_ranks, [1, 2, 3, 4, 4, 5])
assert list(order[1:3]) == [3, 4]  # equal scores retain original order



3. Sorting, searching and ranking
Stable descending scores / indices: [95. 92. 92. 85. 78. 60.] [5 3 4 2 1 0]
Top-3 scores: [95. 92. 92.]
Third smallest (zero-based k=2): 85.0
Search 80 (insertion index, exists): (2, False)
Average ranks (smallest is 1): [1.  2.  3.  4.5 4.5 6. ]
Dense ranks: [1 2 3 4 4 5]
Score quartiles: [79.75 88.5  92.  ]


## 4. Statistics and bounded CSV streaming

axis=0 reduces rows, returning one statistic per feature. nan_policy='omit' ignores NaNs for raw feature means. Score mean is 83.667 and population variance is 143.556. Histogram counts are [1, 1, 1, 3]; quantiles use fractions in [0,1], whereas percentiles use [0,100]. Reading two CSV rows at a time and updating RunningStats agrees with the batch mean and variance.


In [15]:
print("\n4. Statistics and bounded CSV streaming")
print("Score mean / population variance:", nc.mean(scores), nc.var(scores))
print("Score summary:", nc.describe(scores))
counts, edges = nc.histogram(scores, bins=[60, 70, 80, 90, 100])
print("Histogram counts / edges:", counts, edges)
print("Quantiles 0.25 / 0.5 / 0.75:", nc.quantile(scores, [0.25, 0.5, 0.75]))
print("Raw feature means, omit NaNs:", nc.mean(X, axis=0, nan_policy="omit"))
running = nc.RunningStats(ddof=0)
reader = nc.CSVReader(csv_path, dtype=schema, skip_header=1)
chunk_shapes = []
for chunk in reader.iter_chunks(2):
    chunk_shapes.append(chunk.shape)
    running.update(chunk["score"].reshape(-1))
print("Chunk shapes:", chunk_shapes)
print("Streaming count / mean / variance:", running.count, running.mean, running.var)
np.testing.assert_allclose([running.mean, running.var], [nc.mean(scores), nc.var(scores)])
np.testing.assert_array_equal(counts, [1, 1, 1, 3])



4. Statistics and bounded CSV streaming
Score mean / population variance: 83.66666666666667 143.55555555555554
Score summary: {'count': 6, 'mean': 83.66666666666667, 'std': 13.125039682479693, 'min': 60.0, '25%': 79.75, '50%': 88.5, '75%': 92.0, 'max': 95.0}
Histogram counts / edges: [1 1 1 3] [ 60.  70.  80.  90. 100.]
Quantiles 0.25 / 0.5 / 0.75: [79.75 88.5  92.  ]
Raw feature means, omit NaNs: [ 3.5 78. ]
Chunk shapes: [(2, 1), (2, 1), (2, 1)]
Streaming count / mean / variance: 6 83.66666666666667 143.55555555555554


## 5. Estimator Pipeline and metrics

The teaching estimator thresholds the first feature at its fitted mean. Pipeline.fit and predict connect preprocessing to the estimator. Five of six predictions match: accuracy=5/6, precision=1, recall=3/4, F1=6/7. The confusion matrix has true labels on rows and predictions on columns: [[2,0],[1,3]]. These are illustrative in-sample metrics. Continuous hours scores give ROC AUC=1 because they order all positive labels above negative labels, even though the particular hard threshold misses one positive.


In [16]:
class MeanThresholdClassifier:
    """Teaching estimator: learn the first feature's mean, then threshold it.

    Demonstrates Pipeline.fit/predict. The metrics use this same six-row
    example, not a held-out model evaluation.
    """

    def fit(self, X, y=None):
        self.threshold_ = float(np.mean(np.asarray(X)[:, 0]))
        return self

    def predict(self, X):
        return (np.asarray(X)[:, 0] >= self.threshold_).astype(int)

print("\n5. Pipeline.fit / predict and metrics (illustrative, same data)")
model = nc.Pipeline([
    ("impute", nc.Imputer(fill_value=75.0)),
    ("scale", nc.StandardScaler()),
    ("classifier", MeanThresholdClassifier()),
])
model.fit(X, y_true)
y_pred = model.predict(X)
print("Actual / predicted labels:", y_true, y_pred)
print("Accuracy:", nc.accuracy(y_true, y_pred))
print("Precision / recall / F1:", nc.precision(y_true, y_pred),
      nc.recall(y_true, y_pred), nc.f1(y_true, y_pred))
print("Confusion matrix (rows=true, columns=predicted):\n",
      nc.confusion_matrix(y_true, y_pred, labels=[0, 1]))
print("MSE example:", nc.mse([1, 2, 3], [1, 2, 4]))
print("ROC AUC (continuous hours score):", nc.roc_auc_score(y_true, prepared[:, 0]))
np.testing.assert_array_equal(y_pred, [0, 0, 0, 1, 1, 1])
np.testing.assert_allclose(nc.accuracy(y_true, y_pred), 5 / 6)
np.testing.assert_array_equal(nc.confusion_matrix(y_true, y_pred), [[2, 0], [1, 3]])



5. Pipeline.fit / predict and metrics (illustrative, same data)
Actual / predicted labels: [0 0 1 1 1 1] [0 0 0 1 1 1]
Accuracy: 0.8333333333333334
Precision / recall / F1: 1.0 0.75 0.8571428571428571
Confusion matrix (rows=true, columns=predicted):
 [[2 0]
 [1 3]]
MSE example: 0.3333333333333333
ROC AUC (continuous hours score): 1.0


## 6. Finite differences and numerical stability

For f(z)=sum(z**2), the analytic gradient is 2z; at [1,2], the numerical result is approximately [2,4]. For F(z)=[sum(z),sum(z**2)], the Jacobian is [[1,1],[2,4]], with one row per output. Central differences use f(x+h) and f(x-h). Softmax subtracts the maximum before exponentiation, keeping logits near 1000 finite; its probabilities sum to 1.


In [17]:
print("\n6. Gradients, Jacobian and stable utilities")
point = np.array([1.0, 2.0])
grad = nc.gradient(lambda z: np.sum(z ** 2), point, method="central")
jac = nc.jacobian(lambda z: np.array([z.sum(), np.sum(z ** 2)]), point)
print("Gradient of sum(z**2) at [1,2]:", grad)
print("Jacobian of [sum(z), sum(z**2)]:\n", jac)
logits = np.array([[1000.0, 1001.0, 1002.0]])
probabilities = nc.softmax(logits, axis=1)
print("Stable softmax at large logits:", probabilities)
print("Stable logsumexp:", nc.logsumexp(logits, axis=1))
print("Euclidean distance [0,0] to [3,4]:", nc.euclidean_distance([0, 0], [3, 4]))
print("Batch shapes:", [batch.shape for batch in nc.batch_iterator(prepared, 4)])
np.testing.assert_allclose(grad, [2, 4], atol=1e-7)
np.testing.assert_allclose(jac, [[1, 1], [2, 4]], atol=1e-7)
assert np.isfinite(probabilities).all()
np.testing.assert_allclose(probabilities.sum(axis=1), [1])



6. Gradients, Jacobian and stable utilities
Gradient of sum(z**2) at [1,2]: [2. 4.]
Jacobian of [sum(z), sum(z**2)]:
 [[1. 1.]
 [2. 4.]]
Stable softmax at large logits: [[0.09  0.245 0.665]]
Stable logsumexp: [1002.408]
Euclidean distance [0,0] to [3,4]: 5.0
Batch shapes: [(4, 2), (2, 2)]


## 7. Reproducible performance comparison

Fixed seed 42; 20,000 rows; four features for scaling and softmax. Check equivalent results before timing. Report median time per call over five repeats, three calls per repeat and one warmup. Preparation and fitting are excluded. Speed-up is Python baseline / NumCompute; timings vary. Top-k compares a full Python sort to partial selection, so the algorithm also contributes to the difference. The standalone benchmark writes an environment-stamped performance table.


In [18]:
print("\n7. Loop versus NumPy; correctness checked before timing")
display(Markdown(format_table(run_benchmarks())))
print("Median per call; seed=42, 5 repeats, 3 calls, one warmup.")
print("Data preparation and fitting excluded; top-k baseline uses a full Python sort.")
print("\nDemo completed: all output checks passed.")



7. Loop versus NumPy; correctness checked before timing


| Task | Data size | Python baseline (ms) | NumCompute (ms) | Speed-up | Same result |
|---|---|---:|---:|---:|:---:|
| StandardScaler.transform | 20,000 x 4 | 14.4842 | 0.1929 | 75.10x | yes |
| stable softmax | 20,000 x 4 | 22.1580 | 1.7450 | 12.70x | yes |
| top-k | n=20,000, k=10 | 1.9694 | 0.0550 | 35.81x | yes |
| accuracy | n=20,000 | 0.8659 | 0.0222 | 39.00x | yes |

Median per call; seed=42, 5 repeats, 3 calls, one warmup.
Data preparation and fitting excluded; top-k baseline uses a full Python sort.

Demo completed: all output checks passed.


## Tests

Run `python -m pytest -q` for the unit and integration tests. `python demo/verify_demo.py` executes this notebook with a fresh kernel from the project root and demo directory.
